# NovaPay - Respuesta a incidente (DFIR)
Colab es Linux: la Opción B corre aquí solo en **simulación** (dry-run). `--execute` requiere el servidor Windows real.

## Opción B (ganadora): contención SOAR

In [ ]:
%%writefile opcion_b_contencion_soar.py
#!/usr/bin/env python3
"""Opción B - Contención automatizada (SOAR/EDR) para el servidor de conciliación.

Diseño:
  * Simulación (dry-run) por defecto; solo actúa con --execute.
  * Orden: evidencia -> aislar red -> matar conector -> deshabilitar cuenta -> bloquear carpeta.
  * Todo es reversible (rollback) y queda registrado en un log JSONL.
  * Sin shell=True; entradas validadas con regex para evitar inyección de comandos.
Requiere Windows + PowerShell + privilegios de administrador para --execute.
"""
import argparse          # CLI
import csv               # parseo de tasklist
import hashlib           # hash del binario del conector
import io                # buffer para csv
import json              # log estructurado
import re                # validación de entradas
import subprocess        # ejecución de comandos sin shell
import sys               # salida/códigos de retorno
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

# Validadores estrictos: solo caracteres esperables en cada campo.
RE_PROC = re.compile(r"^[A-Za-z0-9_.\-]{1,64}\.exe$")
RE_ACCOUNT = re.compile(r"^[A-Za-z0-9_.\-\\$@]{1,64}$")
RE_IP = re.compile(r"^\d{1,3}(\.\d{1,3}){3}$")


@dataclass
class Config:
    """Parámetros del incidente."""
    process_name: str       # ejecutable del conector PagoLink
    service_account: str    # cuenta de servicio a revocar (DOMINIO\\cuenta)
    folder: str             # carpeta de conciliación
    mgmt_ip: str            # única IP de gestión que conserva acceso
    evidence_dir: Path      # destino de evidencias y backups
    execute: bool           # False = solo simular


class Recorder:
    """Registra cada acción con marca UTC en JSONL (cadena de custodia básica)."""

    def __init__(self, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)   # crea carpeta de evidencias
        self.path = path

    def log(self, step: str, **data):
        rec = {"utc": datetime.now(timezone.utc).isoformat(), "step": step, **data}
        with self.path.open("a", encoding="utf-8") as fh:  # append: nunca sobrescribe
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print(f"[{rec['utc']}] {step} {data}")


def validate(cfg: Config):
    """Rechaza configuraciones que podrían inyectar argumentos."""
    if not RE_PROC.match(cfg.process_name):
        raise ValueError("process_name inválido")
    if not RE_ACCOUNT.match(cfg.service_account):
        raise ValueError("service_account inválido")
    if not RE_IP.match(cfg.mgmt_ip) or any(int(o) > 255 for o in cfg.mgmt_ip.split(".")):
        raise ValueError("mgmt_ip inválida")
    if not cfg.folder or any(c in cfg.folder for c in '"\n\r;|&`'):
        raise ValueError("folder inválido")


def run(cfg: Config, rec: Recorder, step: str, argv: list, timeout=60) -> str:
    """Ejecuta (o simula) un comando; nunca lanza: devuelve '' si falla."""
    if not cfg.execute:
        rec.log(step, mode="dry-run", cmd=argv)
        return ""
    try:
        p = subprocess.run(argv, capture_output=True, text=True, timeout=timeout)
        rec.log(step, rc=p.returncode, stdout=p.stdout[-2000:], stderr=p.stderr[-500:])
        return p.stdout
    except (OSError, subprocess.TimeoutExpired) as exc:   # binario ausente o cuelgue
        rec.log(step, error=str(exc))
        return ""


def ps(cfg, rec, step, script):
    """Atajo para invocar PowerShell sin perfil ni interacción."""
    return run(cfg, rec, step, ["powershell", "-NoProfile", "-NonInteractive", "-Command", script])


# ---------------------------------------------------------------- pasos ----
def collect_evidence(cfg, rec):
    """Captura volátil mínima ANTES de contener (procesos, conexiones, ACL, hash)."""
    d = cfg.evidence_dir
    ps(cfg, rec, "evid_proc", f"Get-CimInstance Win32_Process -Filter \"Name='{cfg.process_name}'\" | "
       f"Select ProcessId,ExecutablePath,CommandLine,CreationDate | ConvertTo-Json | Out-File '{d}\\proc.json'")
    run(cfg, rec, "evid_netstat", ["cmd", "/c", f"netstat -ano > \"{d}\\netstat.txt\""])
    run(cfg, rec, "evid_acl", ["icacls", cfg.folder, "/save", str(d / "acl_backup.txt"), "/t", "/c"])
    if cfg.execute:  # hash del binario para IoC/atribución
        out = ps(cfg, rec, "evid_path", f"(Get-Process {cfg.process_name[:-4]} -ErrorAction SilentlyContinue | "
                 "Select -First 1).Path")
        exe = Path(out.strip()) if out.strip() else None
        if exe and exe.is_file():
            rec.log("evid_sha256", path=str(exe), sha256=hashlib.sha256(exe.read_bytes()).hexdigest())


def isolate_network(cfg, rec):
    """Política de firewall bloquear-todo, con excepción solo para la IP de gestión."""
    run(cfg, rec, "fw_backup", ["netsh", "advfirewall", "export", str(cfg.evidence_dir / "fw_backup.wfw")])
    run(cfg, rec, "fw_allow_in", ["netsh", "advfirewall", "firewall", "add", "rule", "name=IR-ALLOW-MGMT-IN",
        "dir=in", "action=allow", f"remoteip={cfg.mgmt_ip}", "enable=yes", "profile=any"])
    run(cfg, rec, "fw_allow_out", ["netsh", "advfirewall", "firewall", "add", "rule", "name=IR-ALLOW-MGMT-OUT",
        "dir=out", "action=allow", f"remoteip={cfg.mgmt_ip}", "enable=yes", "profile=any"])
    run(cfg, rec, "fw_block_all", ["netsh", "advfirewall", "set", "allprofiles", "firewallpolicy",
        "blockinbound,blockoutbound"])


def kill_connector(cfg, rec):
    """Termina el conector y su árbol de procesos (evidencia ya capturada)."""
    out = run(cfg, rec, "list_proc", ["tasklist", "/FO", "CSV", "/NH", "/FI", f"IMAGENAME eq {cfg.process_name}"])
    for row in csv.reader(io.StringIO(out)):
        if len(row) > 1 and row[1].isdigit():                       # columna PID
            run(cfg, rec, "kill", ["taskkill", "/PID", row[1], "/F", "/T"])
    if not cfg.execute:
        run(cfg, rec, "kill", ["taskkill", "/IM", cfg.process_name, "/F", "/T"])


def revoke_account(cfg, rec):
    """Deshabilita la cuenta (AD si está disponible; si no, cuenta local)."""
    acct = cfg.service_account
    ps(cfg, rec, "disable_account",
       f"if (Get-Command Disable-ADAccount -ErrorAction SilentlyContinue) "
       f"{{ Disable-ADAccount -Identity '{acct.split(chr(92))[-1]}' }} "
       f"else {{ net user '{acct.split(chr(92))[-1]}' /active:no }}")


def lock_folder(cfg, rec):
    """Niega escritura/borrado a la cuenta de servicio; conserva lectura para forense."""
    run(cfg, rec, "acl_deny", ["icacls", cfg.folder, "/deny", f"{cfg.service_account}:(OI)(CI)(W,D,DC)"])


def rollback(cfg, rec):
    """Revierte firewall y ACL (la cuenta se rehabilita manualmente tras rotar credenciales)."""
    run(cfg, rec, "fw_restore", ["netsh", "advfirewall", "import", str(cfg.evidence_dir / "fw_backup.wfw")])
    run(cfg, rec, "acl_restore", ["icacls", str(Path(cfg.folder).parent), "/restore",
                                  str(cfg.evidence_dir / "acl_backup.txt")])


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--process", required=True, help="p.ej. PagoLinkConnector.exe")
    ap.add_argument("--account", required=True, help="DOMINIO\\svc_pagolink")
    ap.add_argument("--folder", required=True)
    ap.add_argument("--mgmt-ip", required=True)
    ap.add_argument("--evidence-dir", default="evidencia_novapay")
    ap.add_argument("--execute", action="store_true", help="aplicar cambios reales")
    ap.add_argument("--rollback", action="store_true")
    a = ap.parse_args()
    cfg = Config(a.process, a.account, a.folder, a.mgmt_ip, Path(a.evidence_dir), a.execute)
    try:
        validate(cfg)
        rec = Recorder(cfg.evidence_dir / "acciones.jsonl")
    except (ValueError, OSError) as exc:
        sys.exit(f"Configuración inválida: {exc}")
    if cfg.execute and sys.platform != "win32":
        sys.exit("--execute requiere Windows")
    cfg.evidence_dir.mkdir(parents=True, exist_ok=True)
    if a.rollback:
        rollback(cfg, rec)
        return
    for step in (collect_evidence, isolate_network, kill_connector, revoke_account, lock_folder):
        try:
            step(cfg, rec)                      # un fallo no debe detener la contención
        except Exception as exc:                # noqa: BLE001 - robustez ante fallos parciales
            rec.log(step.__name__, error=repr(exc))
    rec.log("done", mode="EXECUTE" if cfg.execute else "DRY-RUN")


if __name__ == "__main__":
    main()


In [ ]:
!python opcion_b_contencion_soar.py --process PagoLinkConnector.exe --account 'NOVAPAY\svc_pagolink' --folder 'D:\Conciliacion' --mgmt-ip 10.0.0.5 --evidence-dir evidencia_novapay
!cat evidencia_novapay/acciones.jsonl | head -5

## Opción A: detector de logs (con datos de prueba)

In [ ]:
%%writefile opcion_a_detector_logs.py
#!/usr/bin/env python3
"""Opción A - Detector de exfiltración y modificación masiva (solo detección).

Entradas:
  --net   CSV de red:        timestamp_iso,src_ip,dst_ip,dst_port,bytes_out
  --fs    CSV de archivos:   timestamp_iso,path,action,account
  --allow archivo con IPs/CIDR documentados (uno por línea, # comentarios)
"""
import argparse, csv, ipaddress, sys
from collections import defaultdict, deque
from datetime import datetime, timedelta

MAX_FILES_PER_MIN = 50  # umbral definido en el caso


def load_allowlist(path):
    """Lee redes permitidas; falla cerrado si el archivo no existe."""
    nets = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.split("#")[0].strip()
            if line:
                nets.append(ipaddress.ip_network(line, strict=False))
    return nets


def is_documented(ip, nets):
    """True si la IP es privada/loopback o está en la lista documentada."""
    addr = ipaddress.ip_address(ip)
    return addr.is_private or addr.is_loopback or any(addr in n for n in nets)


def detect_exfil(net_csv, nets):
    """Devuelve alertas por conexiones salientes a destinos no documentados."""
    alerts, totals = [], defaultdict(int)
    with open(net_csv, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            try:
                if not is_documented(row["dst_ip"], nets):
                    totals[(row["src_ip"], row["dst_ip"])] += int(row["bytes_out"])
                    alerts.append(("EXFIL", row["timestamp_iso"], row["src_ip"],
                                   row["dst_ip"], row["dst_port"]))
            except (KeyError, ValueError) as exc:
                print(f"[warn] fila de red inválida: {exc}", file=sys.stderr)
    return alerts, totals


def detect_mass_change(fs_csv):
    """Ventana deslizante de 60 s: alerta si supera MAX_FILES_PER_MIN eventos."""
    window, alerts, fired = deque(), [], False
    with open(fs_csv, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            try:
                ts = datetime.fromisoformat(row["timestamp_iso"])
            except (KeyError, ValueError) as exc:
                print(f"[warn] fila de archivos inválida: {exc}", file=sys.stderr)
                continue
            window.append(ts)
            while window and ts - window[0] > timedelta(seconds=60):
                window.popleft()
            if len(window) > MAX_FILES_PER_MIN and not fired:
                alerts.append(("MASS_CHANGE", ts.isoformat(), len(window), row.get("account", "?")))
                fired = True  # una alerta por ráfaga
            elif len(window) <= MAX_FILES_PER_MIN:
                fired = False
    return alerts


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--net", required=True)
    ap.add_argument("--fs", required=True)
    ap.add_argument("--allow", required=True)
    a = ap.parse_args()
    try:
        nets = load_allowlist(a.allow)
        exfil, totals = detect_exfil(a.net, nets)
        mass = detect_mass_change(a.fs)
    except OSError as exc:
        sys.exit(f"Error de E/S: {exc}")
    for al in exfil + mass:
        print("ALERTA", *al)
    for (src, dst), b in totals.items():
        print(f"RESUMEN {src} -> {dst}: {b} bytes")
    sys.exit(2 if (exfil or mass) else 0)


if __name__ == "__main__":
    main()


In [ ]:
rows = ["timestamp_iso,path,action,account"] + [f"2026-09-30T18:02:{i%50:02d},f{i},mod,svc" for i in range(60)]
open('fs.csv','w').write("\n".join(rows)+"\n")
open('net.csv','w').write("timestamp_iso,src_ip,dst_ip,dst_port,bytes_out\n2026-09-30T17:48:00,10.0.0.9,8.8.8.8,443,9000000\n")
open('allow.txt','w').write("10.0.0.0/8\n")
!python opcion_a_detector_logs.py --net net.csv --fs fs.csv --allow allow.txt

## Opción C: FIM en tiempo real + Sigma
`watchdog` detecta y reacciona; no bloquea escrituras.

In [ ]:
!pip -q install watchdog

In [ ]:
%%writefile opcion_c_fim_realtime.py
#!/usr/bin/env python3
"""Opción C - FIM en tiempo real con watchdog (pip install watchdog).

Nota: watchdog NO puede bloquear escrituras (el kernel ya las aceptó); detecta
y reacciona. Para bloqueo real se requiere ACL/minifiltro/EDR (ver Opción B).
"""
import argparse, subprocess, sys, time, threading
from collections import deque
from watchdog.events import FileSystemEventHandler
from watchdog.observers import Observer

LIMIT, WINDOW = 50, 60.0


class MassChangeHandler(FileSystemEventHandler):
    def __init__(self, on_trigger):
        self.events, self.lock, self.on_trigger, self.tripped = deque(), threading.Lock(), on_trigger, False

    def on_any_event(self, event):
        if event.is_directory:
            return
        now = time.monotonic()
        with self.lock:
            self.events.append(now)
            while self.events and now - self.events[0] > WINDOW:
                self.events.popleft()
            if len(self.events) > LIMIT and not self.tripped:
                self.tripped = True
                self.on_trigger(len(self.events), event.src_path)


def react(count, path, deny_account, folder, execute):
    print(f"[ALERTA] {count} cambios/min; último: {path}", flush=True)
    cmd = ["icacls", folder, "/deny", f"{deny_account}:(OI)(CI)(W,D,DC)"]
    if execute:
        try:
            subprocess.run(cmd, check=True, timeout=30)
        except (OSError, subprocess.SubprocessError) as exc:
            print(f"[error] no se pudo aplicar ACL: {exc}", file=sys.stderr)
    else:
        print("[dry-run]", " ".join(cmd))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--folder", required=True)
    ap.add_argument("--service-account", required=True)
    ap.add_argument("--execute", action="store_true")
    a = ap.parse_args()
    handler = MassChangeHandler(lambda c, p: react(c, p, a.service_account, a.folder, a.execute))
    obs = Observer()
    obs.schedule(handler, a.folder, recursive=True)
    obs.start()
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        pass
    finally:
        obs.stop(); obs.join()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile opcion_c_sigma.yml
title: Conector de conciliación con conexión a destino no documentado
id: 6f1c1a52-0b52-4a2e-9d55-0f6d6f0d1a01
status: experimental
description: Proceso del conector PagoLink conectando a IP pública fuera de la lista documentada
logsource: {category: network_connection, product: windows}
detection:
  selection: {Image|endswith: '\PagoLinkConnector.exe', Initiated: 'true'}
  filter_private:
    DestinationIp|cidr: ['10.0.0.0/8', '172.16.0.0/12', '192.168.0.0/16', '127.0.0.0/8']
  filter_documented:
    DestinationIp|cidr: ['203.0.113.10/32']   # REEMPLAZAR con destinos documentados de PagoLink
  condition: selection and not filter_private and not filter_documented
level: high
tags: [attack.exfiltration, attack.t1041, attack.initial_access, attack.t1195.002]
---
title: Evento de modificación de archivo en carpeta de conciliación
name: novapay_recon_file_change
status: experimental
logsource: {category: file_event, product: windows}
detection:
  selection: {TargetFilename|contains: '\Conciliacion\'}
  condition: selection
---
title: Modificación masiva (>50/min) en carpeta de conciliación
id: 6f1c1a52-0b52-4a2e-9d55-0f6d6f0d1a02
status: experimental
correlation:
  type: event_count
  rules: [novapay_recon_file_change]
  group-by: [Computer, User]
  timespan: 1m
  condition: {gt: 50}
level: critical
tags: [attack.impact, attack.t1486, attack.t1565.001]
